<a href="https://colab.research.google.com/github/profdasigu/manim-google-colab/blob/main/avatar_voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1. VERIFICAR GPU
# Antes de executar:
# Ambiente de execução > Alterar tipo de ambiente > GPU T4
# ============================================================

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

O esperado é algo como:

Tesla T4, 15360 MiB, ...

In [ ]:
# ============================================================
# 2. INSTALAR PYTHON 3.10
# ============================================================

!apt-get update -qq
!apt-get install -y python3.10 python3.10-venv python3.10-dev

!python3.10 --version

In [ ]:
# ============================================================
# 3. CRIAR AMBIENTE VIRTUAL
# ============================================================

!python3.10 -m venv /content/sadtalker-env

!/content/sadtalker-env/bin/python -m pip install --upgrade pip wheel
!/content/sadtalker-env/bin/pip install "setuptools<81"

print("Ambiente criado.")

In [ ]:
!/content/sadtalker-env/bin/python --version
!/content/sadtalker-env/bin/pip --version

Python 3.10.20
pip 26.2.1 from /content/sadtalker-env/lib/python3.10/site-packages/pip (python 3.10)


Verificar se instalou o Python 3.10.20.

In [ ]:
# ============================================================
# 4. BAIXAR SADTALKER
# ============================================================

%cd /content
!rm -rf SadTalker

!git clone https://github.com/OpenTalker/SadTalker.git

%cd /content/SadTalker

print("SadTalker baixado.")

In [ ]:
# ============================================================
# 5. INSTALAR PYTORCH + CUDA
# ============================================================

!/content/sadtalker-env/bin/pip install \
  torch==2.1.2 \
  torchvision==0.16.2 \
  torchaudio==2.1.2 \
  --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# ============================================================
# 6. TESTAR PYTORCH E GPU
# ============================================================

!/content/sadtalker-env/bin/python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA disponível:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SEM GPU')"

O resultado esperado:

PyTorch: 2.1.2+cu121
CUDA: 12.1
CUDA disponível: True
GPU: Tesla T4

In [ ]:
# ============================================================
# 7. INSTALAR FFMPEG
# ============================================================

!apt-get update -qq
!apt-get install -y ffmpeg

!ffmpeg -version | head -1

In [ ]:
# ============================================================
# 8. INSTALAR DEPENDÊNCIAS DO SADTALKER
# ============================================================

%cd /content/SadTalker

!/content/sadtalker-env/bin/pip install -r requirements.txt

# Necessário para compatibilidade do librosa/pkg_resources
!/content/sadtalker-env/bin/pip install "setuptools<81"

In [ ]:
# ============================================================
# 9. TESTAR DEPENDÊNCIAS
# ============================================================

import subprocess

codigo = """
import torch
import cv2
import numpy
import scipy
import librosa

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("OpenCV:", cv2.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Librosa:", librosa.__version__)
print("Tudo OK!")
"""

resultado = subprocess.run(
    ["/content/sadtalker-env/bin/python", "-c", codigo],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.stderr:
    print(resultado.stderr)

In [ ]:
# ============================================================
# 10. BAIXAR MODELOS PRÉ-TREINADOS
# ============================================================

%cd /content/SadTalker

print("Baixando os modelos do SadTalker...")

!rm -rf checkpoints
!bash scripts/download_models.sh

print("Download concluído.")

In [ ]:
!find /content/SadTalker/checkpoints -type f -printf "%f  %s bytes\n"

In [ ]:
Devem aparecer arquivos como:

SadTalker_V0.0.2_512.safetensors
SadTalker_V0.0.2_256.safetensors
mapping_00109-model.pth.tar
mapping_00229-model.pth.tar

In [ ]:
# ============================================================
# 11. ENVIAR IMAGEM DO AVATAR
# Aceita PNG ou JPG
# ============================================================

from google.colab import files
import os
import shutil

os.makedirs(
    "/content/SadTalker/examples/source_image",
    exist_ok=True
)

print("Selecione a imagem do avatar:")

uploaded = files.upload()

arquivo = list(uploaded.keys())[0]
extensao = os.path.splitext(arquivo)[1].lower()

imagem = "/content/SadTalker/examples/source_image/avatar" + extensao

shutil.move(arquivo, imagem)

print("Imagem pronta:")
print(imagem)

In [ ]:
# ============================================================
# 12. ENVIAR ÁUDIO
# Pode enviar diretamente o M4A do Clipchamp
# ============================================================

from google.colab import files
import os
import shutil

os.makedirs(
    "/content/SadTalker/examples/driven_audio",
    exist_ok=True
)

print("Selecione o arquivo de áudio (.m4a):")

uploaded = files.upload()

arquivo = list(uploaded.keys())[0]

audio_m4a = "/content/SadTalker/examples/driven_audio/audio.m4a"

shutil.move(arquivo, audio_m4a)

print("Áudio enviado:")
print(audio_m4a)

In [ ]:
# ============================================================
# 12. ENVIAR E PREPARAR O ÁUDIO
#
# Formatos aceitos:
# - M4A (Clipchamp)
# - OGG (WhatsApp)
# - MP3
# - WAV
#
# O arquivo será convertido automaticamente para WAV
# compatível com o SadTalker.
# ============================================================

from google.colab import files
import os
import subprocess

pasta_audio = "/content/SadTalker/examples/driven_audio"
os.makedirs(pasta_audio, exist_ok=True)

print("Selecione o arquivo de áudio:")
print("Formatos aceitos: M4A, OGG, MP3 ou WAV")

uploaded = files.upload()

arquivo_original = list(uploaded.keys())[0]

extensao = os.path.splitext(arquivo_original)[1].lower()

formatos_aceitos = [".m4a", ".ogg", ".mp3", ".wav"]

if extensao not in formatos_aceitos:
    raise ValueError(
        f"Formato {extensao} não suportado. "
        "Envie M4A, OGG, MP3 ou WAV."
    )

audio_wav = os.path.join(pasta_audio, "audio.wav")

print("\nConvertendo áudio...")

resultado = subprocess.run([
    "ffmpeg",
    "-y",
    "-i", arquivo_original,
    "-ac", "1",
    "-ar", "16000",
    audio_wav
], capture_output=True, text=True)

if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError("Erro na conversão do áudio.")

print("\n✓ Áudio preparado com sucesso!")
print("Arquivo:", audio_wav)

In [ ]:
# ============================================================
# 14. GERAR VÍDEO COM SADTALKER
# ============================================================

%cd /content/SadTalker

!rm -rf ./results
!mkdir -p ./results

!MPLBACKEND=Agg /content/sadtalker-env/bin/python inference.py \
  --driven_audio ./examples/driven_audio/audio.wav \
  --source_image {imagem} \
  --result_dir ./results \
  --still \
  --preprocess full \
  --enhancer gfpgan

In [ ]:
# ============================================================
# 15. VISUALIZAR O VÍDEO
# ============================================================

from IPython.display import Video, display
import glob
import os

videos = glob.glob(
    "/content/SadTalker/results/**/*.mp4",
    recursive=True
)

if videos:

    video_final = max(
        videos,
        key=os.path.getmtime
    )

    print("Vídeo gerado:")
    print(video_final)

    display(
        Video(
            video_final,
            embed=True,
            width=600
        )
    )

else:

    print("Nenhum vídeo foi encontrado.")

In [ ]:
# ============================================================
# 16. BAIXAR O VÍDEO
# ============================================================

from google.colab import files

files.download(video_final)